In [96]:
import pandas as pd
import json

In [97]:
df_terms_final = pd.read_parquet('cleaned_aws_terms.parquet')
df_products = pd.read_parquet('cleaned_aws_products.parquet')

In [98]:
print("Terms shape:", df_terms_final.shape)
print("Products shape:", df_products.shape)
df_products.head(3)

Terms shape: (99979, 5)
Products shape: (100000, 44)


,sku,productFamily,attributes.servicecode,attributes.location,attributes.locationType,attributes.instanceType,attributes.currentGeneration,attributes.instanceFamily,attributes.vcpu,attributes.physicalProcessor,attributes.memory,attributes.storage,attributes.networkPerformance,attributes.processorArchitecture,attributes.engineCode,attributes.databaseEngine,attributes.databaseEdition,attributes.licenseModel,attributes.deploymentOption,attributes.usagetype,attributes.operation,attributes.engineMediaType,attributes.instanceTypeFamily,attributes.normalizationSizeFactor,attributes.regionCode,attributes.servicename,attributes.unbundledLicensing,attributes.windowslicensemultiplier,attributes.clockSpeed,attributes.dedicatedEbsThroughput,attributes.enhancedNetworkingSupported,attributes.volumeType,attributes.group,attributes.groupDescription,attributes.volumeName,attributes.deploymentModel,attributes.engineMajorVersion,attributes.extendedSupportPricingYear,attributes.processorFeatures,attributes.storageMedia,attributes.minVolumeSize,attributes.maxVolumeSize,attributes.limitlesspreview,attributes.acu
0,SJCPEREKG4AJC3P7,Database Instance,AmazonRDS,Canada (Central),AWS Region,db.r8i.xlarge,Yes,Memory optimized,4,Intel Xeon Scalable (Granite Rapids),32 GiB,EBS Only,Up to 12500 Megabit,x86,52,SQL Server,Standard,Bring your own media,Multi-AZ,CAN1-MirrorUsage:db.r8i.xl,CreateDBInstance:0052,Customer-provided,R8i,NA,ca-central-1,Amazon Relational Database Service,TRUE,2,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
1,JYX35YJ8Z5DXXS98,Database Instance,AmazonRDS,Asia Pacific (Mumbai),AWS Region,db.r5d.24xlarge,Yes,Memory optimized,96,Intel Xeon Platinum 8175,768 GiB,4 x 900 NVMe SSD,25 Gbps,64-bit,2,MySQL,None,No license required,Single-AZ,APS3-InstanceUsage:db.r5d.24xl,CreateDBInstance:0002,None,R5d,192,ap-south-1,Amazon Relational Database Service,FALSE,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
2,H4WDB4TZHXB8URCU,Database Instance,AmazonRDS,Asia Pacific (Melbourne),AWS Region,db.t3.medium,Yes,General purpose,2,Intel Skylake E5 2686 v5 (2.5 GHz),4 GiB,EBS Only,Low to Moderate,64-bit,2,MySQL,None,No license required,Multi-AZ,APS6-Multi-AZUsage:db.t3.medium,CreateDBInstance:0002,None,T3,4,ap-southeast-4,Amazon Relational Database Service,FALSE,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None


In [99]:
if 'attributes.servicename' in df_products.columns:
    df_products.drop(columns=['attributes.servicename'], errors='ignore', inplace=True)

print (f"Dimensions (rows, cols): {df_products.shape}")

Dimensions (rows, cols): (100000, 43)


In [100]:
df_database_instance= df_products[df_products['productFamily'] == 'Database Instance'].reset_index(drop=True)

target_storage_families = ['Database Storage', 'Provisioned IOPS']
df_rds_storage = df_products[df_products['productFamily'].isin(target_storage_families)].reset_index(drop=True)

excluded_families = ['Database Instance', 'Database Storage', 'Provisioned IOPS']
df_rds_extras = df_products[~df_products['productFamily'].isin(excluded_families)].reset_index(drop=True)


#### Ανάλυση Database Instance

In [101]:
id_columns_db_instance = ['sku', 'productFamily', 
    'attributes.servicecode',
    'attributes.location', 
    'attributes.locationType']

In [102]:
feature_columns_db_instance = ['attributes.instanceType', 'attributes.instanceFamily', 
                                'attributes.vcpu', 'attributes.memory', 
                                'attributes.storage', 'attributes.physicalProcessor', 
                                'attributes.networkPerformance', 'attributes.databaseEngine', 
                                'attributes.databaseEdition', 'attributes.licenseModel', 
                                'attributes.deploymentOption']

In [103]:
total_db_instance = []
total_db_instance.extend(id_columns_db_instance)
total_db_instance.extend(feature_columns_db_instance)
df_database_instance[total_db_instance].head()

,sku,productFamily,attributes.servicecode,attributes.location,attributes.locationType,attributes.instanceType,attributes.instanceFamily,attributes.vcpu,attributes.memory,attributes.storage,attributes.physicalProcessor,attributes.networkPerformance,attributes.databaseEngine,attributes.databaseEdition,attributes.licenseModel,attributes.deploymentOption
0,SJCPEREKG4AJC3P7,Database Instance,AmazonRDS,Canada (Central),AWS Region,db.r8i.xlarge,Memory optimized,4,32 GiB,EBS Only,Intel Xeon Scalable (Granite Rapids),Up to 12500 Megabit,SQL Server,Standard,Bring your own media,Multi-AZ
1,JYX35YJ8Z5DXXS98,Database Instance,AmazonRDS,Asia Pacific (Mumbai),AWS Region,db.r5d.24xlarge,Memory optimized,96,768 GiB,4 x 900 NVMe SSD,Intel Xeon Platinum 8175,25 Gbps,MySQL,None,No license required,Single-AZ
2,H4WDB4TZHXB8URCU,Database Instance,AmazonRDS,Asia Pacific (Melbourne),AWS Region,db.t3.medium,General purpose,2,4 GiB,EBS Only,Intel Skylake E5 2686 v5 (2.5 GHz),Low to Moderate,MySQL,None,No license required,Multi-AZ
3,R8Q2435NXQCAMNX9,Database Instance,AmazonRDS,US East (Ohio),AWS Region,db.x1e.8xlarge,Memory optimized,32,976 GiB,1 x 960 SSD,Intel Xeon E7-8880 v3,Up to 10 Gigabit,Oracle,Enterprise,Bring your own license,Multi-AZ
4,U86UU5KVR2Z9BH2T,Database Instance,AmazonRDS,Africa (Cape Town),AWS Region,db.r5.4xlarge,Memory optimized,16,128 GiB,Aurora IO Optimization Mode,Intel Xeon Platinum 8175,Up to 10 Gigabit,Aurora PostgreSQL,None,No license required,Single-AZ


In [104]:
remaining_cols_rds = [col for col in df_database_instance.columns if col not in total_db_instance]

df_final_database_instance = df_database_instance[total_db_instance].copy()

df_final_database_instance['additional_attributes'] = df_database_instance[remaining_cols_rds].apply(
    lambda row: json.dumps({k: v for k, v in row.to_dict().items() if pd.notna(v)}), 
    axis=1
)

In [105]:
df_master_database_instance = pd.merge(
    df_final_database_instance, 
    df_terms_final, 
    on='sku', 
    how='inner'
)

df_master_database_instance = df_master_database_instance.reset_index(drop=True)

In [106]:
pd.set_option('display.max_colwidth', 50)
pd.set_option('display.max_columns', None)
df_master_database_instance.head()

,sku,productFamily,attributes.servicecode,attributes.location,attributes.locationType,attributes.instanceType,attributes.instanceFamily,attributes.vcpu,attributes.memory,attributes.storage,attributes.physicalProcessor,attributes.networkPerformance,attributes.databaseEngine,attributes.databaseEdition,attributes.licenseModel,attributes.deploymentOption,additional_attributes,rateCode,description,unit,priceUSD
0,SJCPEREKG4AJC3P7,Database Instance,AmazonRDS,Canada (Central),AWS Region,db.r8i.xlarge,Memory optimized,4,32 GiB,EBS Only,Intel Xeon Scalable (Granite Rapids),Up to 12500 Megabit,SQL Server,Standard,Bring your own media,Multi-AZ,"{""attributes.currentGeneration"": ""Yes"", ""attri...",SJCPEREKG4AJC3P7.JRTCKXETXF.6YS6EN2CT7,USD 1.08 per db.r8i.xlarge Multi-AZ instance h...,Hrs,1.0800
1,JYX35YJ8Z5DXXS98,Database Instance,AmazonRDS,Asia Pacific (Mumbai),AWS Region,db.r5d.24xlarge,Memory optimized,96,768 GiB,4 x 900 NVMe SSD,Intel Xeon Platinum 8175,25 Gbps,MySQL,None,No license required,Single-AZ,"{""attributes.currentGeneration"": ""Yes"", ""attri...",JYX35YJ8Z5DXXS98.JRTCKXETXF.6YS6EN2CT7,$ 15.89 per RDS db.r5d.24xlarge Single-AZ inst...,Hrs,15.8900
2,H4WDB4TZHXB8URCU,Database Instance,AmazonRDS,Asia Pacific (Melbourne),AWS Region,db.t3.medium,General purpose,2,4 GiB,EBS Only,Intel Skylake E5 2686 v5 (2.5 GHz),Low to Moderate,MySQL,None,No license required,Multi-AZ,"{""attributes.currentGeneration"": ""Yes"", ""attri...",H4WDB4TZHXB8URCU.JRTCKXETXF.6YS6EN2CT7,USD 0.224 db.t3.medium Multi-AZ instance hour ...,Hrs,0.2240
3,R8Q2435NXQCAMNX9,Database Instance,AmazonRDS,US East (Ohio),AWS Region,db.x1e.8xlarge,Memory optimized,32,976 GiB,1 x 960 SSD,Intel Xeon E7-8880 v3,Up to 10 Gigabit,Oracle,Enterprise,Bring your own license,Multi-AZ,"{""attributes.currentGeneration"": ""Yes"", ""attri...",R8Q2435NXQCAMNX9.JRTCKXETXF.6YS6EN2CT7,USD 22.418 per RDS db.x1e.8xlarge Multi-AZ ins...,Hrs,22.4179
4,U86UU5KVR2Z9BH2T,Database Instance,AmazonRDS,Africa (Cape Town),AWS Region,db.r5.4xlarge,Memory optimized,16,128 GiB,Aurora IO Optimization Mode,Intel Xeon Platinum 8175,Up to 10 Gigabit,Aurora PostgreSQL,None,No license required,Single-AZ,"{""attributes.currentGeneration"": ""Yes"", ""attri...",U86UU5KVR2Z9BH2T.JRTCKXETXF.6YS6EN2CT7,USD 3.962 per RDS db.r5.4xlarge IO-optimized S...,Hrs,3.9620


#### Ανάλυση Storage και IOPS

In [107]:
id_columns_storage = ['sku', 'productFamily', 
                    'attributes.servicecode', 
                    'attributes.location', 
                    'attributes.locationType']

In [108]:
feature_columns_storage = ['attributes.volumeName', 'attributes.volumeType',          
                            'attributes.storageMedia', 'attributes.minVolumeSize',      
                            'attributes.maxVolumeSize', 'attributes.databaseEngine',     
                            'attributes.deploymentOption', 'attributes.usagetype']

In [109]:
total_storage_cols = id_columns_storage + feature_columns_storage

remaining_cols_storage = [col for col in df_rds_storage.columns if col not in total_storage_cols]

df_final_storage = df_rds_storage[total_storage_cols].copy()

df_final_storage['additional_attributes'] = df_rds_storage[remaining_cols_storage].apply(
    lambda row: json.dumps({k: v for k, v in row.to_dict().items() if pd.notna(v)}), 
    axis=1
)

In [110]:
df_master_storage = pd.merge(
    df_final_storage, 
    df_terms_final, 
    on='sku', 
    how='inner'
)
df_master_storage = df_master_storage.reset_index(drop=True)


In [111]:
pd.set_option('display.max_colwidth', 50)
pd.set_option('display.max_columns', None)
df_master_storage.head()

,sku,productFamily,attributes.servicecode,attributes.location,attributes.locationType,attributes.volumeName,attributes.volumeType,attributes.storageMedia,attributes.minVolumeSize,attributes.maxVolumeSize,attributes.databaseEngine,attributes.deploymentOption,attributes.usagetype,additional_attributes,rateCode,description,unit,priceUSD
0,4PM8P98M785XKUYK,Provisioned IOPS,AmazonRDS,EU (Stockholm),AWS Region,gp3,General Purpose-GP3,None,None,None,Db2,Single-AZ,EUN1-RDS:GP3-PIOPS,"{""attributes.engineCode"": ""29"", ""attributes.da...",4PM8P98M785XKUYK.JRTCKXETXF.6YS6EN2CT7,USD 0.021 per IOPS-month of provisioned GP3 IO...,IOPS-Mo,0.021
1,RR5W6T9RM7MZ5V4Y,Provisioned IOPS,AmazonRDS,US West (Oregon),AWS Region,None,Provisioned IOPS-IO2,None,None,None,Db2,Multi-AZ,USW2-RDS:Multi-AZ-IO2-PIOPS,"{""attributes.engineCode"": ""27"", ""attributes.da...",RR5W6T9RM7MZ5V4Y.JRTCKXETXF.6YS6EN2CT7,USD 0.2 per IOPS-Month of provisioned io2 IOPS...,IOPS-Mo,0.200
2,DTRVMZGG9K7XE2WH,Provisioned IOPS,AmazonRDS,US West (Oregon),AWS Region,None,None,None,None,None,Oracle,Multi-AZ,USW2-RDS:Multi-AZ-PIOPS,"{""attributes.engineCode"": ""20"", ""attributes.da...",DTRVMZGG9K7XE2WH.JRTCKXETXF.6YS6EN2CT7,$0.2 per IOPS-month of provisioned io1 IOPS fo...,IOPS-Mo,0.200
3,TWKAWDVADK96EWBM,Database Storage,AmazonRDS,Asia Pacific (Sydney),AWS Region,gp2,General Purpose (SSD),None,None,None,SQL Server,Multi-AZ,APS2-RDS:Multi-AZ-GP2-Storage,"{""attributes.engineCode"": ""402"", ""attributes.d...",TWKAWDVADK96EWBM.JRTCKXETXF.6YS6EN2CT7,USD 0.276 per GB-month of provisioned gp2 stor...,GB-Mo,0.276
4,53V4VKYJ3PXTYMM2,Database Storage,AmazonRDS,Asia Pacific (Singapore),AWS Region,None,General Purpose-GP3,SSD,5 GB,16 TB,Any,Multi-AZ (SQL Server Mirror),APS1-RDS:Mirror-GP3-Storage,"{""attributes.engineCode"": ""0"", ""attributes.ope...",53V4VKYJ3PXTYMM2.JRTCKXETXF.6YS6EN2CT7,$0.276 per GB-month of provisioned GP3 storage...,GB-Mo,0.276


#### Ανάλυση υπολοίπων πεδίων

In [112]:
id_columns_extras = ['sku', 'productFamily', 
                    'attributes.servicecode', 
                    'attributes.location', 
                    'attributes.locationType']

In [113]:
feature_columns_extras = ['attributes.usagetype','attributes.operation',
                        'attributes.databaseEngine','attributes.engineMajorVersion',
                        'attributes.extendedSupportPricingYear','attributes.group',
                        'attributes.acu']

In [114]:
total_extras_cols = id_columns_extras + feature_columns_extras

remaining_cols_extras = [col for col in df_rds_extras.columns if col not in total_extras_cols]

df_final_extras = df_rds_extras[total_extras_cols].copy()
df_final_extras['additional_attributes'] = df_rds_extras[remaining_cols_extras].apply(
    lambda row: json.dumps({k: v for k, v in row.to_dict().items() if pd.notna(v)}), 
    axis=1
)

In [115]:
df_master_extras = pd.merge(
    df_final_extras, 
    df_terms_final, 
    on='sku', 
    how='inner'
).reset_index(drop=True)

In [116]:
pd.set_option('display.max_colwidth', 50)
pd.set_option('display.max_columns', None)
df_master_extras.head()

,sku,productFamily,attributes.servicecode,attributes.location,attributes.locationType,attributes.usagetype,attributes.operation,attributes.databaseEngine,attributes.engineMajorVersion,attributes.extendedSupportPricingYear,attributes.group,attributes.acu,additional_attributes,rateCode,description,unit,priceUSD
0,4B2HN9YDXBGUNJCS,ServerlessV2,AmazonRDS,EU (Paris),AWS Region,EUW3-Aurora:ServerlessV2Usage,CreateDBInstance:0021,Aurora PostgreSQL,None,None,None,None,"{""attributes.engineCode"": ""21"", ""attributes.re...",4B2HN9YDXBGUNJCS.JRTCKXETXF.6YS6EN2CT7,USD 0.14 per Aurora Capacity Unit hour running...,ACU-Hr,0.1400
1,6K8P2MJZWHR2K36S,Performance Insights,AmazonRDS,Canada West (Calgary),AWS Region,CAN2:PI_LTR_FMR:Provisioned,CreateDBInstance:0011,SQL Server,None,None,None,None,"{""attributes.engineCode"": ""11"", ""attributes.in...",6K8P2MJZWHR2K36S.JRTCKXETXF.6YS6EN2CT7,$1.6098 per vCPU-month for one month retentio...,vCPU-Months,1.6098
2,T2WKWGYA2NH3WYKV,None,AmazonRDS,Asia Pacific (Malaysia),AWS Region,APS7-ExtendedSupport:Yr3:AuroraPostgreSQL11,CreateDBInstance:0021,Aurora PostgreSQL,11,Year 3,None,None,"{""attributes.engineCode"": ""21"", ""attributes.re...",T2WKWGYA2NH3WYKV.JRTCKXETXF.6YS6EN2CT7,USD 0.216 per hour per vCPU running RDS Extend...,vCPU-hour,0.2160
3,5FRN26GK46ZNUS7D,Provisioned Throughput,AmazonRDS,EU (Frankfurt),AWS Region,EUC1-RDS:Mirror-GP3-Throughput,CreateDBInstance,Any,None,None,RDS-Throughput,None,"{""attributes.engineCode"": ""0"", ""attributes.dep...",5FRN26GK46ZNUS7D.JRTCKXETXF.6YS6EN2CT7,$0.191 per MiBps-month of provisioned GP3 Sto...,MBPS-Mo,0.1910
4,64QUNA4MWJE7GZUZ,Provisioned Throughput,AmazonRDS,Europe (Spain),AWS Region,EUS2-RDS:Multi-AZ-GP3-Throughput,CreateDBInstance:0035,Db2,None,None,RDS-Throughput,None,"{""attributes.engineCode"": ""35"", ""attributes.da...",64QUNA4MWJE7GZUZ.JRTCKXETXF.6YS6EN2CT7,USD 0.177 per MiBps-month of provisioned GP3 ...,MBPS-Mo,0.1770


#### Κώδικας εντοπισμού στηλών

In [117]:
summary_data = []
for col in df_rds_extras.columns:
    sample_vals = list(df_rds_extras[col].dropna().unique()[:5])
    summary_data.append({'Column': col, 'Sample_Values': sample_vals})

df_summary = pd.DataFrame(summary_data)

# Εμφάνιση χωρίς περικοπές
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
df_summary

,Column,Sample_Values
0,sku,"[4B2HN9YDXBGUNJCS, 6K8P2MJZWHR2K36S, T2WKWGYA2NH3WYKV, 5FRN26GK46ZNUS7D, 64QUNA4MWJE7GZUZ]"
1,productFamily,"[ServerlessV2, Performance Insights, Provisioned Throughput, RDSProxy, CPU Credits]"
2,attributes.servicecode,[AmazonRDS]
3,attributes.location,"[EU (Paris), Canada West (Calgary), Asia Pacific (Malaysia), EU (Frankfurt), Europe (Spain)]"
4,attributes.locationType,"[AWS Region, AWS Outposts]"
5,attributes.instanceType,[]
6,attributes.currentGeneration,[]
7,attributes.instanceFamily,"[T3, T4G]"
8,attributes.vcpu,"[1, 0]"
9,attributes.physicalProcessor,[]
